# Assignment 1 — training pipeline

## 1. Imports and Configuration

In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import torch

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,
)

In [2]:
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_PATH = Path("data/shopping.csv")
ARTIFACT_DIR = Path("artifacts")

ARTIFACT_DIR.mkdir(exist_ok=True)

print("Setup complete.")
print("Dataset exists:", DATA_PATH.exists())

Setup complete.
Dataset exists: True


## 2. Load and Inspect Data

In [ ]:
df = pd.read_csv(DATA_PATH)

Shape: (12330, 18)

Columns:
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend', 'Revenue']

Missing values:
Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser                    0
Region                     0
TrafficType                0
VisitorType                0
Weekend                    0
Revenue                    0
dtype: int64

Data types:
Administrative               int64
Administrative_Duration    float64
Informational                int64
Informati

In [ ]:
print("Shape:", df.shape)

In [ ]:
print("Columns:")
print(df.columns.tolist())

In [ ]:
print("Missing values:")
print(df.isna().sum())

In [ ]:
print("Data types:")
print(df.dtypes)

## 3. Target Definition

This converts our target (Revenue) to 0/1.

In [ ]:
df["Revenue"] = df["Revenue"].astype(int)

## 4. Feature Engineering

In [ ]:
MONTH_MAP = {"Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6, "June": 6, "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12,}

In [ ]:
def engineer_features(data):
    data = data.copy()

    # All features used here are available at session entry.
    data["Weekend"] = data["Weekend"].astype(int)

    data["MonthNum"] = data["Month"].map(MONTH_MAP)

    if data["MonthNum"].isna().any():
        raise ValueError("Unknown month values in input")

    # Cyclical calendar representation
    data["MonthSin"] = np.sin(
        2 * np.pi * data["MonthNum"] / 12
    )

    data["MonthCos"] = np.cos(
        2 * np.pi * data["MonthNum"] / 12
    )

    # Visitor status is assumed known when the session starts.
    is_returning = (
        data["VisitorType"] == "Returning_Visitor"
    ).astype(int)

    # Interactions between visitor history and calendar context
    data["ReturningWeekend"] = (
        is_returning * data["Weekend"]
    )

    data["ReturningSpecialDay"] = (
        is_returning * data["SpecialDay"]
    )

    # Categorical interaction: visitor type and traffic source
    data["TrafficVisitorSegment"] = (
        data["TrafficType"].astype(str)
        + "__"
        + data["VisitorType"].astype(str)
    )

    return data

  Month  MonthNum  MonthSin  MonthCos  ReturningWeekend  ReturningSpecialDay  \
0   Feb         2  0.866025       0.5                 0                  0.0   
1   Feb         2  0.866025       0.5                 0                  0.0   
2   Feb         2  0.866025       0.5                 0                  0.0   
3   Feb         2  0.866025       0.5                 0                  0.0   
4   Feb         2  0.866025       0.5                 1                  0.0   

  TrafficVisitorSegment  
0  1__Returning_Visitor  
1  2__Returning_Visitor  
2  3__Returning_Visitor  
3  4__Returning_Visitor  
4  4__Returning_Visitor  


In [ ]:
df_model = engineer_features(df)
print(df_model[["Month", "MonthNum", "MonthSin", "MonthCos", "ReturningWeekend", "ReturningSpecialDay", "TrafficVisitorSegment",]].head())

## 5. Chronological Train/Val/Test Split

In [6]:
train_df = df_model[df_model["MonthNum"] <= 10].copy()
val_df = df_model[df_model["MonthNum"] == 11].copy()
test_df = df_model[df_model["MonthNum"] == 12].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (7605, 24)
Validation: (2998, 24)
Test: (1727, 24)


### 5.1 Sanity checks

In [7]:
# Sanity checks for the chronological split
assert len(train_df) > 0, "Training set is empty."
assert len(val_df) > 0, "Validation set is empty."
assert len(test_df) > 0, "Test set is empty."

# Each split should contain both classes
assert train_df["Revenue"].nunique() == 2, "Training set does not contain both classes."
assert val_df["Revenue"].nunique() == 2, "Validation set does not contain both classes."
assert test_df["Revenue"].nunique() == 2, "Test set does not contain both classes."

# Verify chronological ordering
assert train_df["MonthNum"].max() <= 10
assert val_df["MonthNum"].eq(11).all()
assert test_df["MonthNum"].eq(12).all()

### 5.2 Class checks

In [8]:
print("\nPositive class rates:")
print(f"Train:      {train_df['Revenue'].mean():.3f}")
print(f"Validation: {val_df['Revenue'].mean():.3f}")

print("\nClass counts:")
print("Train:")
print(train_df["Revenue"].value_counts().sort_index())

print("\nValidation:")
print(val_df["Revenue"].value_counts().sort_index())


Positive class rates:
Train:      0.123
Validation: 0.254

Class counts:
Train:
Revenue
0    6673
1     932
Name: count, dtype: int64

Validation:
Revenue
0    2238
1     760
Name: count, dtype: int64


## 6. Feature Selection

In [ ]:
# Entry-time original predictors
ENTRY_NUMERIC = ["SpecialDay", "Weekend",]

ENTRY_CATEGORICAL = ["OperatingSystems", "Browser", "Region", "TrafficType", "VisitorType",]

# Engineered entry-time predictors
CALENDAR_ENGINEERED = ["MonthSin", "MonthCos",]
VISITOR_ENGINEERED = ["ReturningWeekend", "ReturningSpecialDay",]

# Proposed final feature specification
NUMERIC_FEATURES = (ENTRY_NUMERIC + CALENDAR_ENGINEERED + VISITOR_ENGINEERED)
CATEGORICAL_FEATURES = (ENTRY_CATEGORICAL + ["TrafficVisitorSegment"])
TARGET = "Revenue"


# Guard against completed-session information entering the model.
UNAVAILABLE_AT_ENTRY = {
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
}

assert not UNAVAILABLE_AT_ENTRY.intersection(NUMERIC_FEATURES + CATEGORICAL_FEATURES)

assert TARGET not in (NUMERIC_FEATURES + CATEGORICAL_FEATURES)

print("Entry-time predictors:", NUMERIC_FEATURES + CATEGORICAL_FEATURES,)


Entry-time predictors: ['SpecialDay', 'Weekend', 'MonthSin', 'MonthCos', 'ReturningWeekend', 'ReturningSpecialDay', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'TrafficVisitorSegment']


## 7. Preprocessing and Feature Transformation

In [10]:
X_train_raw = train_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
X_val_raw = val_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
X_test_raw = test_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]

y_train = train_df[TARGET].to_numpy(dtype=np.int64)
y_val = val_df[TARGET].to_numpy(dtype=np.int64)
y_test = test_df[TARGET].to_numpy(dtype=np.int64)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            NUMERIC_FEATURES,
        ),
        (
            "categorical",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=False,
            ),
            CATEGORICAL_FEATURES,
        ),
    ]
)

X_train = preprocessor.fit_transform(X_train_raw)
X_val = preprocessor.transform(X_val_raw)
X_test = preprocessor.transform(X_test_raw)

feature_names = preprocessor.get_feature_names_out()

Model features:
- numeric__SpecialDay
- numeric__Weekend
- numeric__MonthSin
- numeric__MonthCos
- numeric__ReturningWeekend
- numeric__ReturningSpecialDay
- categorical__OperatingSystems_2
- categorical__OperatingSystems_3
- categorical__OperatingSystems_4
- categorical__OperatingSystems_5
- categorical__OperatingSystems_6
- categorical__OperatingSystems_7
- categorical__OperatingSystems_8
- categorical__Browser_2
- categorical__Browser_3
- categorical__Browser_4
- categorical__Browser_5
- categorical__Browser_6
- categorical__Browser_7
- categorical__Browser_8
- categorical__Browser_9
- categorical__Browser_10
- categorical__Browser_11
- categorical__Browser_12
- categorical__Browser_13
- categorical__Region_2
- categorical__Region_3
- categorical__Region_4
- categorical__Region_5
- categorical__Region_6
- categorical__Region_7
- categorical__Region_8
- categorical__Region_9
- categorical__TrafficType_2
- categorical__TrafficType_3
- categorical__TrafficType_4
- categorical__TrafficT

/Users/gonzalo/daiist-assignment-1/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/gonzalo/daiist-assignment-1/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [ ]:
print("Model features:")
for name in feature_names:
    print("-", name)

X_train = np.asarray(X_train, dtype=np.float64)
X_val = np.asarray(X_val, dtype=np.float64)
X_test = np.asarray(X_test, dtype=np.float64)

assert X_train.shape[1] == len(feature_names)
assert X_val.shape[1] == X_train.shape[1]
assert X_test.shape[1] == X_train.shape[1]

assert np.isfinite(X_train).all()
assert np.isfinite(X_val).all()
assert np.isfinite(X_test).all()

In [12]:
print("Processed shapes:")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("Number of model features:", X_train.shape[1])

Processed shapes:
Train: (7605, 92)
Validation: (2998, 92)
Test: (1727, 92)
Number of model features: 92


## 8. Evaluation Metrics and Business Costs

These are assumed, illustrative service-level costs

In [ ]:
ASSISTANCE_COST = 1.0
MISSED_BUYER_SERVICE_COST = 5.0

FP_COST = ASSISTANCE_COST
TP_COST = ASSISTANCE_COST
FN_COST = MISSED_BUYER_SERVICE_COST

### 8.1 Evaluation

In [ ]:
def evaluate_model(y_true, probabilities, threshold=0.5):

    if not 0 <= threshold <= 1:
        raise ValueError("threshold must be between 0 and 1")

    probabilities = np.asarray(probabilities, dtype=float).reshape(-1)

    y_true = np.asarray(y_true, dtype=int).reshape(-1)

    if len(y_true) != len(probabilities):
        raise ValueError("Targets and probabilities must have same length")

    predictions = (probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1],).ravel()

    contacted = int(tp + fp)

    # Service-level scenario cost:
    # every contact consumes staff resources;
    # each unassisted eventual buyer carries an assumed penalty.
    
    business_cost = ((ASSISTANCE_COST * contacted) + (MISSED_BUYER_SERVICE_COST * fn))

    return {
        "accuracy": accuracy_score(y_true, predictions),

        "precision": precision_score(y_true, predictions, zero_division=0),

        "recall": recall_score(y_true, predictions, zero_division=0),

        "f1": f1_score(y_true, predictions, zero_division=0),

        "roc_auc": roc_auc_score(y_true, probabilities),

        "log_loss": log_loss(y_true, probabilities, labels=[0, 1]),

        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),

        "contacted": contacted,
        "contact_rate": contacted / len(y_true),

        "business_cost": float(business_cost),
    }


## 9. Naive Baseline

In [14]:
# Naive baseline:
# predict the training-set purchase rate for every observation
train_purchase_rate = float(y_train.mean())

baseline_val_prob = np.full(
    len(y_val),
    train_purchase_rate,
    dtype=np.float64,
)

baseline_test_prob = np.full(
    len(y_test),
    train_purchase_rate,
    dtype=np.float64,
)

baseline_val_metrics = evaluate_model(
    y_val,
    baseline_val_prob,
    threshold=0.5,
)

print(f"Training purchase rate: {train_purchase_rate:.4f}")

print("\nNaive baseline — validation:")
print(baseline_val_metrics)

Training purchase rate: 0.1226

Naive baseline — validation:
{'accuracy': 0.7464976651100734, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'roc_auc': 0.5, 'log_loss': 0.6297537095501908, 'tn': 2238, 'fp': 0, 'fn': 760, 'tp': 0, 'contacted': 0, 'contact_rate': 0.0, 'business_cost': 3800.0}


## 10. Scikit-learn Logistic Regression

In [ ]:
from sklearn.base import clone

REGULARIZATION_CANDIDATES = [0.1, 1.0]
REGULARIZATION_MONTHS = [7, 8]

regularization_rows = []

for candidate_c in REGULARIZATION_CANDIDATES:
    for month in REGULARIZATION_MONTHS:

        earlier = df_model[df_model["MonthNum"] < month]
        later = df_model[df_model["MonthNum"] == month]

        columns = NUMERIC_FEATURES + CATEGORICAL_FEATURES

        fold_preprocessor = clone(preprocessor)

        fold_X_train = fold_preprocessor.fit_transform(earlier[columns]).astype(np.float64)

        fold_X_val = fold_preprocessor.transform(later[columns]).astype(np.float64)

        fold_model = LogisticRegression(
            C=candidate_c,
            solver="lbfgs",
            max_iter=3000,
        )

        fold_model.fit(
            fold_X_train,
            earlier[TARGET],
        )

        fold_probabilities = fold_model.predict_proba(fold_X_val)[:, 1]

        regularization_rows.append({
            "C": candidate_c,
            "month": month,
            "log_loss": log_loss(
                later[TARGET],
                fold_probabilities,
            ),
        })

regularization_df = pd.DataFrame(regularization_rows)

regularization_summary = (regularization_df.groupby("C")["log_loss"].mean())

print(regularization_summary)

C
0.1    0.457140
1.0    0.471032
Name: log_loss, dtype: float64
Chosen C: 0.1


/Users/gonzalo/daiist-assignment-1/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1, 5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/gonzalo/daiist-assignment-1/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1, 5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [ ]:
SKLEARN_C = float(regularization_summary.idxmin())

print("Chosen C:", SKLEARN_C)

In [16]:
sklearn_model = LogisticRegression(
    C=SKLEARN_C,
    solver="lbfgs",
    max_iter=3000,
    tol=1e-9,
)

sklearn_model.fit(
    X_train,
    y_train,
)

sklearn_val_prob = sklearn_model.predict_proba(X_val)[:, 1]

## 11. Manual PyTorch Logistic Regression

In [17]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float64)
X_val_tensor = torch.tensor(X_val, dtype=torch.float64)
X_test_tensor = torch.tensor(X_test, dtype=torch.float64)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.float64,
).reshape(-1, 1)

y_val_tensor = torch.tensor(
    y_val,
    dtype=torch.float64,
).reshape(-1, 1)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.float64,
).reshape(-1, 1)

print("X_train tensor:", X_train_tensor.shape)
print("y_train tensor:", y_train_tensor.shape)

X_train tensor: torch.Size([7605, 92])
y_train tensor: torch.Size([7605, 1])


In [ ]:
MANUAL_LR = 1.0
MANUAL_EPOCHS = 5000

n_features = X_train_tensor.shape[1]

manual_weights = torch.zeros((n_features, 1),
    dtype=torch.float64,
    requires_grad=True,
)

manual_bias = torch.zeros(1,
    dtype=torch.float64,
    requires_grad=True,
)

Weights: torch.Size([92, 1])
Bias: torch.Size([1])


In [ ]:
print("Weights:", manual_weights.shape)
print("Bias:", manual_bias.shape)

In [ ]:
manual_train_losses = []
manual_val_losses = []

for epoch in range(MANUAL_EPOCHS):
    # 1. Forward pass
    train_logits = X_train_tensor @ manual_weights + manual_bias

    # 2. Calculate binary classification loss
    train_loss = torch.nn.functional.binary_cross_entropy_with_logits(
        train_logits,
        y_train_tensor,
    )

    train_loss = (train_loss + manual_weights.square().sum() / (2 * len(y_train) * SKLEARN_C))

    # 3. Ask autograd to calculate gradients
    train_loss.backward()

    # 4. Manually update parameters using gradient descent
    with torch.no_grad():
        manual_weights -= MANUAL_LR * manual_weights.grad
        manual_bias -= MANUAL_LR * manual_bias.grad

    # 5. Clear gradients before the next iteration
    manual_weights.grad.zero_()
    manual_bias.grad.zero_()

    # Record losses
    with torch.no_grad():
        val_logits = X_val_tensor @ manual_weights + manual_bias

        val_loss = torch.nn.functional.binary_cross_entropy_with_logits(val_logits, y_val_tensor,)

    manual_train_losses.append(train_loss.item())
    manual_val_losses.append(val_loss.item())

    if epoch == 0 or (epoch + 1) % 250 == 0:
        print(
            f"Epoch {epoch + 1:4d} | "
            f"train loss = {train_loss.item():.4f} | "
            f"val loss = {val_loss.item():.4f}"
        )

Epoch    1 | train loss = 0.6931 | val loss = 0.5733
Epoch  250 | train loss = 0.3468 | val loss = 0.5823
Epoch  500 | train loss = 0.3463 | val loss = 0.5824
Epoch  750 | train loss = 0.3462 | val loss = 0.5826
Epoch 1000 | train loss = 0.3461 | val loss = 0.5826
Epoch 1250 | train loss = 0.3461 | val loss = 0.5826
Epoch 1500 | train loss = 0.3461 | val loss = 0.5826
Epoch 1750 | train loss = 0.3461 | val loss = 0.5826
Epoch 2000 | train loss = 0.3461 | val loss = 0.5826
Epoch 2250 | train loss = 0.3461 | val loss = 0.5826
Epoch 2500 | train loss = 0.3461 | val loss = 0.5826
Epoch 2750 | train loss = 0.3461 | val loss = 0.5826
Epoch 3000 | train loss = 0.3461 | val loss = 0.5826
Epoch 3250 | train loss = 0.3461 | val loss = 0.5826
Epoch 3500 | train loss = 0.3461 | val loss = 0.5826
Epoch 3750 | train loss = 0.3461 | val loss = 0.5826
Epoch 4000 | train loss = 0.3461 | val loss = 0.5826
Epoch 4250 | train loss = 0.3461 | val loss = 0.5826
Epoch 4500 | train loss = 0.3461 | val loss = 

In [20]:
with torch.no_grad():
    manual_val_logits = (
        X_val_tensor @ manual_weights + manual_bias
    )

    manual_val_prob = torch.sigmoid(
        manual_val_logits
    ).squeeze().numpy()

## 12. Standard Pytorch (nn.Module, torch.optim)

In [ ]:
class TorchLogisticRegression(torch.nn.Module):
    def __init__(self, n_features):
        super().__init__()

        self.linear = torch.nn.Linear(n_features, 1,)

    def forward(self, x):
        return self.linear(x)

In [22]:
torch.manual_seed(SEED)

torch_model = TorchLogisticRegression(
    n_features=X_train.shape[1]
).double()

# Start from zeros, just like our manual model.
with torch.no_grad():
    torch_model.linear.weight.zero_()
    torch_model.linear.bias.zero_()

print(torch_model)

TorchLogisticRegression(
  (linear): Linear(in_features=92, out_features=1, bias=True)
)


In [23]:
TORCH_LR = 1.0
TORCH_EPOCHS = 5000

criterion = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.SGD(
    torch_model.parameters(),
    lr=TORCH_LR,
)

In [24]:
torch_train_losses = []
torch_val_losses = []

for epoch in range(TORCH_EPOCHS):
    torch_model.train()

    # Clear old gradients
    optimizer.zero_grad()

    # Forward pass
    train_logits = torch_model(X_train_tensor)

    # Calculate loss
    train_loss = criterion(
        train_logits,
        y_train_tensor,
    )

    train_loss = (train_loss + torch_model.linear.weight.square().sum() / (2 * len(y_train) * SKLEARN_C))

    # Calculate gradients
    train_loss.backward()

    # Update weights and bias
    optimizer.step()

    # Validation
    torch_model.eval()

    with torch.no_grad():
        val_logits = torch_model(X_val_tensor)

        val_loss = criterion(
            val_logits,
            y_val_tensor,
        )

    torch_train_losses.append(train_loss.item())
    torch_val_losses.append(val_loss.item())

    if epoch == 0 or (epoch + 1) % 250 == 0:
        print(
            f"Epoch {epoch + 1:4d} | "
            f"train loss = {train_loss.item():.4f} | "
            f"val loss = {val_loss.item():.4f}"
        )

Epoch    1 | train loss = 0.6931 | val loss = 0.5733
Epoch  250 | train loss = 0.3468 | val loss = 0.5823
Epoch  500 | train loss = 0.3463 | val loss = 0.5824
Epoch  750 | train loss = 0.3462 | val loss = 0.5826
Epoch 1000 | train loss = 0.3461 | val loss = 0.5826
Epoch 1250 | train loss = 0.3461 | val loss = 0.5826
Epoch 1500 | train loss = 0.3461 | val loss = 0.5826
Epoch 1750 | train loss = 0.3461 | val loss = 0.5826
Epoch 2000 | train loss = 0.3461 | val loss = 0.5826
Epoch 2250 | train loss = 0.3461 | val loss = 0.5826
Epoch 2500 | train loss = 0.3461 | val loss = 0.5826
Epoch 2750 | train loss = 0.3461 | val loss = 0.5826
Epoch 3000 | train loss = 0.3461 | val loss = 0.5826
Epoch 3250 | train loss = 0.3461 | val loss = 0.5826
Epoch 3500 | train loss = 0.3461 | val loss = 0.5826
Epoch 3750 | train loss = 0.3461 | val loss = 0.5826
Epoch 4000 | train loss = 0.3461 | val loss = 0.5826
Epoch 4250 | train loss = 0.3461 | val loss = 0.5826
Epoch 4500 | train loss = 0.3461 | val loss = 

In [25]:
torch_model.eval()

with torch.no_grad():
    torch_val_logits = torch_model(X_val_tensor)

    torch_val_prob = torch.sigmoid(
        torch_val_logits
    ).squeeze().numpy()

## 13. Model Convergence

In [ ]:
# DIAGNOSTIC: Compare the training objectives of all three models.
# This does not modify or retrain the models.

sklearn_train_prob = sklearn_model.predict_proba(X_train)[:, 1]

torch_model.eval()

with torch.no_grad():
    manual_train_prob = torch.sigmoid(
        X_train_tensor @ manual_weights + manual_bias
    ).reshape(-1).cpu().numpy()

    module_train_prob = torch.sigmoid(
        torch_model(X_train_tensor)
    ).reshape(-1).cpu().numpy()


model_predictions = [
    (
        "Scikit-learn",
        sklearn_train_prob,
        sklearn_val_prob,
    ),
    (
        "Manual PyTorch",
        manual_train_prob,
        manual_val_prob,
    ),
    (
        "PyTorch nn.Module",
        module_train_prob,
        torch_val_prob,
    ),
]

print("THREE-MODEL OPTIMIZATION DIAGNOSTIC")

for name, train_prob, val_prob in model_predictions:
    print(
        f"{name:20s}: "
        f"train log loss: {log_loss(y_train, train_prob)} / "
        f"validation log loss: {log_loss(y_val, val_prob)}"
    )

difference = np.abs(
    sklearn_train_prob - manual_train_prob
)

print("Scikit-learn iterations:", sklearn_model.n_iter_)
print("Mean training probability difference:", difference.mean())
print("95th percentile difference:", np.quantile(difference, 0.95))
print("Maximum training probability difference:", difference.max())



=== THREE-MODEL OPTIMIZATION DIAGNOSTIC ===
Scikit-learn         | train log loss: 0.344358 | validation log loss: 0.582616
Manual PyTorch       | train log loss: 0.344359 | validation log loss: 0.582616
PyTorch nn.Module    | train log loss: 0.344359 | validation log loss: 0.582616

Scikit-learn iterations: [84]
Mean training probability difference: 4.068261229970063e-06
95th percentile difference: 1.2270411100157017e-05
Maximum training probability difference: 0.0001545738978684652


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    manual_train_losses,
    label="Training objective (BCE + L2)",
)

plt.plot(
    manual_val_losses,
    label="Validation BCE (no regularization penalty)",
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Manual PyTorch — training and validation losses")
plt.legend()

plt.tight_layout()

plt.savefig(
    ARTIFACT_DIR / "manual_training_losses.png",
    dpi=150,
    bbox_inches="tight",
)

plt.close()

## 14. Validation and Threshold

In [ ]:
threshold_results = []

thresholds = np.arange(0.05, 0.951, 0.01,)

for threshold in thresholds:
    metrics = evaluate_model(
        y_val,
        sklearn_val_prob,
        threshold=threshold,
    )

    threshold_results.append({"threshold": threshold, **metrics,})

threshold_df = pd.DataFrame(threshold_results)
threshold_df.head()

,threshold,accuracy,precision,recall,f1,roc_auc,log_loss,tn,fp,fn,tp,contacted,contact_rate,business_cost
0,0.05,0.282522,0.259426,0.986842,0.410846,0.604114,0.582616,97,2141,10,750,2891,0.964310,2941.0
1,0.06,0.296197,0.261989,0.977632,0.413237,0.604114,0.582616,145,2093,17,743,2836,0.945964,2921.0
2,0.07,0.334223,0.270260,0.956579,0.421449,0.604114,0.582616,275,1963,33,727,2690,0.897265,2855.0
3,0.08,0.359907,0.274777,0.930263,0.424242,0.604114,0.582616,372,1866,53,707,2573,0.858239,2838.0
4,0.09,0.394263,0.279633,0.881579,0.424588,0.604114,0.582616,512,1726,90,670,2396,0.799199,2846.0


In [ ]:
best_row = threshold_df.loc[threshold_df["business_cost"].idxmin()]

best_threshold = float(best_row["threshold"])

print("November-only cost-minimizing threshold (diagnostic, not final policy):")
print(best_threshold)

print("\nValidation metrics at best threshold:")
print(best_row)

November-only cost-minimizing threshold (diagnostic, not final policy):
0.12000000000000002

Validation metrics at best threshold:
threshold           0.120000
accuracy            0.482322
precision           0.298576
recall              0.772368
f1                  0.430668
roc_auc             0.604114
log_loss            0.582616
tn                859.000000
fp               1379.000000
fn                173.000000
tp                587.000000
contacted        1966.000000
contact_rate        0.655771
business_cost    2831.000000
Name: 7, dtype: float64


In [30]:
default_val_metrics = evaluate_model(
    y_val,
    sklearn_val_prob,
    threshold=0.5,
)

best_val_metrics = evaluate_model(
    y_val,
    sklearn_val_prob,
    threshold=best_threshold,
)

print("Threshold 0.50 business cost:")
print(default_val_metrics["business_cost"])

print("\nNovember-only cost-minimizing threshold cost:")
print(best_val_metrics["business_cost"])

Threshold 0.50 business cost:
3800.0

November-only cost-minimizing threshold cost:
2831.0


In [ ]:
# STEP 1: Diagnose probability quality and business thresholds.
# Do not use the December test set in this experiment.



# 1. Obtain probabilities for the training set.
sklearn_train_prob = sklearn_model.predict_proba(X_train)[:, 1]

# 2. Compare actual and predicted purchase rates.
diagnostic_rows = []

for name, actual, probabilities in [
    ("Training", y_train, sklearn_train_prob),
    ("November validation", y_val, sklearn_val_prob),
]:
    diagnostic_rows.append({
        "period": name,
        "sessions": len(actual),
        "actual_purchase_rate": actual.mean(),
        "mean_predicted_probability": probabilities.mean(),
        "brier_score": brier_score_loss(actual, probabilities),
        "log_loss": log_loss(actual, probabilities),
    })

diagnostic_df = pd.DataFrame(diagnostic_rows)

print("PROBABILITY DIAGNOSTICS")
print(diagnostic_df.round(4).to_string(index=False))


# 3. Compare thresholds using November ONLY.
# Cost-only theoretical threshold:
# contact if the expected missed-service penalty exceeds
# the cost of contacting a visitor.
# Assumes calibrated probabilities and ignores capacity.

theoretical_threshold = (
    ASSISTANCE_COST / MISSED_BUYER_SERVICE_COST
)

threshold_rows = []

for name, threshold in [
    ("Validation-optimized", best_threshold),
    ("Cost-theoretical", theoretical_threshold),
    ("Default", 0.50),
]:
    metrics = evaluate_model(
        y_val,
        sklearn_val_prob,
        threshold=threshold,
    )

    threshold_rows.append({
        "method": name,
        "threshold": threshold,
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "fp": metrics["fp"],
        "fn": metrics["fn"],
        "business_cost": metrics["business_cost"],
        "cost_per_session": metrics["business_cost"] / len(y_val),
    })

threshold_comparison = pd.DataFrame(threshold_rows)

print("NOVEMBER THRESHOLD COMPARISON")
print(threshold_comparison.round(4).to_string(index=False))

print("\nNovember naive-policy costs:")

print(
    "Always predict no purchase:",
    evaluate_model(
        y_val,
        np.zeros(len(y_val)),
        threshold=0.5,
    )["business_cost"],
)

print(
    "Always predict purchase:",
    evaluate_model(
        y_val,
        np.ones(len(y_val)),
        threshold=0.5,
    )["business_cost"],
)


# 4. Examine new vs. returning visitors.
# We are analyzing groups, NOT training separate models.

for name, frame, actual, probabilities in [
    ("Training (in-sample)", train_df, y_train, sklearn_train_prob),
    ("November validation", val_df, y_val, sklearn_val_prob),
]:
    segment_data = pd.DataFrame({
        "visitor_type": frame["VisitorType"].to_numpy(),
        "actual": actual,
        "predicted_probability": probabilities,
    })

    segment_results = segment_data.groupby("visitor_type").agg(
        sessions=("actual", "size"),
        actual_purchase_rate=("actual", "mean"),
        mean_predicted_probability=("predicted_probability", "mean"),
    )

    print(f"{name}: VISITOR TYPES")
    print(segment_results.round(4).to_string())



=== PROBABILITY DIAGNOSTICS ===
             period  sessions  actual_purchase_rate  mean_predicted_probability  brier_score  log_loss
           Training      7605                0.1226                      0.1226       0.1014    0.3444
November validation      2998                0.2535                      0.1636       0.1927    0.5826

=== NOVEMBER THRESHOLD COMPARISON ===
              method  threshold  precision  recall   fp  fn  business_cost  cost_per_session
Validation-optimized       0.12     0.2986  0.7724 1379 173         2831.0            0.9443
    Cost-theoretical       0.20     0.3228  0.4553  726 414         3142.0            1.0480
             Default       0.50     0.0000  0.0000    0 760         3800.0            1.2675

November naive-policy costs:
Always predict no purchase: 3800.0
Always predict purchase: 2238.0

=== Training (in-sample): VISITOR TYPES ===
                   sessions  actual_purchase_rate  mean_predicted_probability
visitor_type               

## 15. Rolling Validation

In [ ]:
# ENTRY-TIME OPERATING POLICY
# Maximize recall of eventual purchasers,
# subject to the assistance-capacity constraint.
# The 20% limit is a hypothetical business assumption.

VALIDATION_MONTHS = [7, 8, 9, 10, 11]

ASSISTANCE_CAPACITY = 0.20

THRESHOLD_GRID = np.round(
    np.arange(0.0, 1.001, 0.01),
    2,
)

In [ ]:
fold_predictions = {}

# Obtain out-of-sample predictions for historical months.
for month in VALIDATION_MONTHS:

    fold_train = df_model.loc[
        df_model["MonthNum"] < month
    ]

    fold_val = df_model.loc[
        df_model["MonthNum"] == month
    ]

    columns = NUMERIC_FEATURES + CATEGORICAL_FEATURES

    fold_preprocessor = clone(preprocessor)

    X_fold_train = np.asarray(
        fold_preprocessor.fit_transform(
            fold_train[columns]
        ),
        dtype=np.float64,
    )

    X_fold_val = np.asarray(
        fold_preprocessor.transform(
            fold_val[columns]
        ),
        dtype=np.float64,
    )

    fold_model = LogisticRegression(
        C=SKLEARN_C,
        solver="lbfgs",
        max_iter=3000,
    )

    fold_model.fit(
        X_fold_train,
        fold_train[TARGET],
    )

    probabilities = fold_model.predict_proba(
        X_fold_val
    )[:, 1]

    fold_predictions[month] = {
        "actual": fold_val[TARGET].to_numpy(dtype=int),
        "probabilities": probabilities,
    }


# Evaluate candidate thresholds across the historical folds.
threshold_scores = []

for threshold in THRESHOLD_GRID:

    monthly_metrics = [
        evaluate_model(
            fold_predictions[month]["actual"],
            fold_predictions[month]["probabilities"],
            threshold=threshold,
        )
        for month in VALIDATION_MONTHS
    ]

    threshold_scores.append({
        "threshold": float(threshold),

        "mean_monthly_recall": float(np.mean([
            m["recall"] for m in monthly_metrics
        ])),

        "mean_monthly_contact_rate": float(np.mean([
            m["contact_rate"] for m in monthly_metrics
        ])),

        "max_monthly_contact_rate": float(np.max([
            m["contact_rate"] for m in monthly_metrics
        ])),

        "mean_monthly_cost": float(np.mean([
            m["business_cost"] / len(
                fold_predictions[month]["actual"]
            )
            for month, m in zip(
                VALIDATION_MONTHS, monthly_metrics
            )
        ])),
    })


rolling_results = pd.DataFrame(threshold_scores)

# Retain thresholds that historically respect capacity
# in EVERY validation month.
feasible = rolling_results.loc[
    rolling_results["max_monthly_contact_rate"]
    <= ASSISTANCE_CAPACITY + 1e-12
].copy()

if feasible.empty:
    raise RuntimeError(
        "No capacity-feasible threshold found"
    )

# Main criterion: maximize historical recall.
# Illustrative cost and threshold resolve ties.
feasible = feasible.sort_values(
    [
        "mean_monthly_recall",
        "mean_monthly_cost",
        "threshold",
    ],
    ascending=[False, True, False],
)

best_threshold = float(
    feasible.iloc[0]["threshold"]
)

print(
    "\n=== POLICY A: CONSERVATIVE HISTORICAL THRESHOLD ==="
)

print(
    feasible.head(10)
    .round(4)
    .to_string(index=False)
)

print(
    "Policy A conservative threshold:",
    best_threshold
)


# Inspect how the frozen policy behaved in each historical month.
comparison_rows = []

for month in VALIDATION_MONTHS:

    fold = fold_predictions[month]

    metrics = evaluate_model(
        fold["actual"],
        fold["probabilities"],
        best_threshold,
    )

    comparison_rows.append({
        "month": month,
        "sessions": len(fold["actual"]),
        "purchase_rate": fold["actual"].mean(),
        "contact_rate": metrics["contact_rate"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "cost_per_session": (
            metrics["business_cost"]
            / len(fold["actual"])
        ),
    })

comparison_df = pd.DataFrame(comparison_rows)

print("HISTORICAL MONTH-BY-MONTH POLICY AUDIT")

print(comparison_df.round(4).to_string(index=False))

print("Note: December may exceed the 20% contact rate. A live rate limiter would be needed for a hard capacity cap.")


/Users/gonzalo/daiist-assignment-1/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1, 5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/gonzalo/daiist-assignment-1/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)



=== POLICY A: CONSERVATIVE HISTORICAL THRESHOLD ===
 threshold  mean_monthly_recall  mean_monthly_contact_rate  max_monthly_contact_rate  mean_monthly_cost
      0.25               0.1426                     0.1141                    0.1853             0.9483
      0.26               0.1303                     0.1026                    0.1741             0.9494
      0.27               0.1157                     0.0888                    0.1585             0.9523
      0.28               0.1017                     0.0770                    0.1562             0.9563
      0.29               0.0797                     0.0575                    0.1228             0.9607
      0.30               0.0575                     0.0427                    0.0982             0.9688
      0.31               0.0475                     0.0329                    0.0848             0.9694
      0.32               0.0380                     0.0260                    0.0692             0.9720
      0.33 

## 16. Historical Policy Comparison

In [ ]:
# HISTORICAL COMPARISON OF THREE ASSISTANCE POLICIES
# No December observations are used for policy selection.

POLICY_A = "A: Conservative fixed cutoff"
POLICY_B = "B: Historical capacity quantile"
POLICY_C = "C: Cost floor + capacity"

POLICY_ORDER = [POLICY_A, POLICY_B, POLICY_C]


def fit_capacity_recall_threshold(history_months):
    """
    POLICY A:
    Find the cutoff with the highest mean monthly recall
    while respecting capacity in each historical month.
    Only earlier labeled validation folds are used.
    """

    candidates = []

    for threshold in THRESHOLD_GRID:

        fold_metrics = [
            evaluate_model(
                fold_predictions[m]["actual"],
                fold_predictions[m]["probabilities"],
                float(threshold),
            )
            for m in history_months
        ]

        maximum_contact_rate = max(
            x["contact_rate"] for x in fold_metrics
        )

        if maximum_contact_rate > ASSISTANCE_CAPACITY + 1e-12:
            continue

        mean_recall = np.mean([
            x["recall"] for x in fold_metrics
        ])

        mean_cost = np.mean([
            x["business_cost"]
            / len(fold_predictions[m]["actual"])
            for m, x in zip(history_months, fold_metrics)
        ])

        # Minimize the negative recall.
        # Cost and then higher thresholds break ties.
        candidates.append((
            -float(mean_recall),
            float(mean_cost),
            -float(threshold),
            float(threshold),
        ))

    if not candidates:
        raise ValueError(
            "No historical threshold respects capacity."
        )

    return min(candidates)[3]


def fit_historical_capacity_cutoff(history_months):
    """
    POLICY B:
    Estimate the upper 20% score boundary using only
    OUT-OF-SAMPLE predictions from previous months.

    The current month's scores are never used to choose
    that month's cutoff.
    """

    previous_scores = np.concatenate([
        fold_predictions[m]["probabilities"]
        for m in history_months
    ])

    boundary = float(np.quantile(
        previous_scores,
        1 - ASSISTANCE_CAPACITY,
        method="higher",
    ))

    # Contact scores strictly above the historical quantile. This also handles tied boundary scores conservatively.
    return min(
        1.0,
        float(np.nextafter(boundary, np.inf)),
    )


def get_policy_cutoffs(history_months):
    """
    Return the three policy thresholds fitted
    using only the supplied historical months.
    """

    cutoff_a = fit_capacity_recall_threshold(
        history_months
    )

    cutoff_b = fit_historical_capacity_cutoff(
        history_months
    )

    # Theoretical expected-cost cutoff:
    # contact cost = 1
    # missed-buyer service penalty = 5

    # This requires calibrated probabilities to have
    # an economic interpretation. It is hypothetical.
    
    cost_floor = (ASSISTANCE_COST / MISSED_BUYER_SERVICE_COST)

    cutoff_c = max(cutoff_b, cost_floor)

    return {
        POLICY_A: cutoff_a,
        POLICY_B: cutoff_b,
        POLICY_C: cutoff_c,
    }


=== POLICY REPLAY: MONTH BY MONTH ===
 month                          policy  threshold  sessions  actual_purchasers  contacted  contact_rate  precision  recall  tp  fn  cost_per_session  capacity_exceeded
     9    A: Conservative fixed cutoff     0.2000       448                 86        169        0.3772     0.2604  0.5116  44  42            0.8460               True
     9 B: Historical capacity quantile     0.1630       448                 86        236        0.5268     0.2331  0.6395  55  31            0.8728               True
     9        C: Cost floor + capacity     0.2000       448                 86        169        0.3772     0.2604  0.5116  44  42            0.8460               True
    10    A: Conservative fixed cutoff     0.2500       549                115         98        0.1785     0.2347  0.2000  23  92            1.0164              False
    10 B: Historical capacity quantile     0.1959       549                115        209        0.3807     0.2727  0.495

In [ ]:
# WALK-FORWARD HISTORICAL POLICY REPLAY

# July and August provide the first calibration history.
# September, October and November evaluate the policies.

POLICY_EVALUATION_MONTHS = [9, 10, 11]

policy_replay_rows = []

for month in POLICY_EVALUATION_MONTHS:

    previous_months = [
        m for m in VALIDATION_MONTHS
        if m < month
    ]

    policy_cutoffs = get_policy_cutoffs(
        previous_months
    )

    fold = fold_predictions[month]

    for policy_name, cutoff in policy_cutoffs.items():

        metrics = evaluate_model(
            fold["actual"],
            fold["probabilities"],
            threshold=cutoff,
        )

        policy_replay_rows.append({
            "month": month,
            "policy": policy_name,
            "threshold": cutoff,

            "sessions": len(fold["actual"]),
            "actual_purchasers": int(
                fold["actual"].sum()
            ),

            "contacted": metrics["contacted"],
            "contact_rate": metrics["contact_rate"],

            "precision": metrics["precision"],
            "recall": metrics["recall"],

            "tp": metrics["tp"],
            "fn": metrics["fn"],

            "cost_per_session": (
                metrics["business_cost"]
                / len(fold["actual"])
            ),

            "capacity_exceeded": (
                metrics["contact_rate"]
                > ASSISTANCE_CAPACITY
            ),
        })


policy_replay_df = pd.DataFrame(
    policy_replay_rows
)

print("\n=== POLICY REPLAY: MONTH BY MONTH ===")

print(
    policy_replay_df
    .round(4)
    .to_string(index=False)
)

In [ ]:
# AGGREGATED HISTORICAL COMPARISON

policy_summary_df = (
    policy_replay_df
    .groupby("policy", sort=False)
    .agg(
        mean_monthly_recall=("recall", "mean"),
        mean_monthly_cost=("cost_per_session", "mean"),

        mean_contact_rate=("contact_rate", "mean"),
        max_contact_rate=("contact_rate", "max"),

        total_tp=("tp", "sum"),
        total_purchasers=("actual_purchasers", "sum"),

        total_contacts=("contacted", "sum"),
        total_sessions=("sessions", "sum"),
    )
    .reset_index()
)

policy_summary_df["pooled_recall"] = (
    policy_summary_df["total_tp"]
    / policy_summary_df["total_purchasers"]
)

policy_summary_df["pooled_contact_rate"] = (
    policy_summary_df["total_contacts"]
    / policy_summary_df["total_sessions"]
)

policy_summary_df["within_capacity_in_replay"] = (
    policy_summary_df["max_contact_rate"]
    <= ASSISTANCE_CAPACITY + 1e-12
)

print("\n=== POLICY REPLAY: SUMMARY ===")

print(
    policy_summary_df
    .round(4)
    .to_string(index=False)
)

In [ ]:
# CHOOSE THE POLICY BEFORE EXAMINING DECEMBER

# First require historical capacity feasibility.
# Then maximize mean monthly buyer recall.
# Use hypothetical cost and simplicity to break ties.

eligible = policy_summary_df.loc[
    policy_summary_df["within_capacity_in_replay"]
].copy()

if eligible.empty:

    print(
        "\nWARNING: No policy respected the 20% capacity "
        "limit in every replay month."
    )

    # Predeclared conservative fallback.
    # This does NOT guarantee December feasibility.
    selected_policy = POLICY_A

else:

    eligible["simplicity_order"] = (
        eligible["policy"].map({
            name: i
            for i, name in enumerate(POLICY_ORDER)
        })
    )

    eligible = eligible.sort_values(
        [
            "mean_monthly_recall",
            "mean_monthly_cost",
            "simplicity_order",
        ],
        ascending=[False, True, True],
    )

    selected_policy = str(
        eligible.iloc[0]["policy"]
    )


# Refit policy thresholds using ALL historical folds,
# July through November. No December data is involved.

final_policy_thresholds = get_policy_cutoffs(
    VALIDATION_MONTHS
)

selected_threshold = float(
    final_policy_thresholds[selected_policy]
)

# Ensure the Policy A reimplementation matches Cell 34.
assert np.isclose(
    final_policy_thresholds[POLICY_A],
    best_threshold,
)

print("\n=== FROZEN DECEMBER POLICY ===")
print("Selected policy:", selected_policy)

print("\nFinal policy thresholds:")

for policy_name, cutoff in final_policy_thresholds.items():
    print(f"{policy_name}: {cutoff:.5f}")

print("\nSelected threshold:", selected_threshold)

print(
    "\nHistorical capacity compliance is not a guarantee "
    "of future capacity compliance."
)

## 17. Feature Engineering - Ablation

In [34]:
# STEP 3: Feature-engineering ablation study.

# Feature-ablation study using entry-time information only.
# December is excluded from feature comparisons.

FEATURE_SETS = {
    "A: Entry baseline": (
        ENTRY_NUMERIC,
        ENTRY_CATEGORICAL,
    ),

    "B: + Cyclical calendar": (
        ENTRY_NUMERIC + CALENDAR_ENGINEERED,
        ENTRY_CATEGORICAL,
    ),

    "C: + Visitor interactions": (
        ENTRY_NUMERIC
        + CALENDAR_ENGINEERED
        + VISITOR_ENGINEERED,
        ENTRY_CATEGORICAL,
    ),

    "D: + Traffic/visitor segment": (
        NUMERIC_FEATURES,
        CATEGORICAL_FEATURES,
    ),
}

ablation_rows = []

for feature_set_name, (
    numeric_cols,
    categorical_cols,
) in FEATURE_SETS.items():

    columns = numeric_cols + categorical_cols

    for month in VALIDATION_MONTHS:

        fold_train = df_model.loc[
            df_model["MonthNum"] < month
        ]

        fold_val = df_model.loc[
            df_model["MonthNum"] == month
        ]

        fold_preprocessor = ColumnTransformer([
            (
                "numeric",
                StandardScaler(),
                numeric_cols,
            ),
            (
                "categorical",
                OneHotEncoder(
                    drop="first",
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
                categorical_cols,
            ),
        ])

        X_fold_train = np.asarray(
            fold_preprocessor.fit_transform(
                fold_train[columns]
            ),
            dtype=np.float64,
        )

        X_fold_val = np.asarray(
            fold_preprocessor.transform(
                fold_val[columns]
            ),
            dtype=np.float64,
        )

        fold_model = LogisticRegression(
            C=SKLEARN_C,
            solver="lbfgs",
            max_iter=3000,
        )

        fold_model.fit(
            X_fold_train,
            fold_train[TARGET],
        )

        probabilities = fold_model.predict_proba(
            X_fold_val
        )[:, 1]

        metrics = evaluate_model(
            fold_val[TARGET],
            probabilities,
            selected_threshold,
        )

        ablation_rows.append({
            "feature_set": feature_set_name,
            "month": month,
            "sessions": len(fold_val),

            "business_cost": metrics["business_cost"],

            "cost_per_session": (
                metrics["business_cost"]
                / len(fold_val)
            ),

            "contact_rate": metrics["contact_rate"],
            "recall": metrics["recall"],
            "roc_auc": metrics["roc_auc"],
            "log_loss": metrics["log_loss"],
        })


ablation_df = pd.DataFrame(ablation_rows)

ablation_summary = (
    ablation_df
    .groupby("feature_set", sort=False)
    .agg(
        total_cost=("business_cost", "sum"),
        total_sessions=("sessions", "sum"),
        mean_monthly_cost=("cost_per_session", "mean"),
        mean_contact_rate=("contact_rate", "mean"),
        mean_recall=("recall", "mean"),
        mean_roc_auc=("roc_auc", "mean"),
        mean_log_loss=("log_loss", "mean"),
    )
)

ablation_summary["pooled_cost_per_session"] = (
    ablation_summary["total_cost"]
    / ablation_summary["total_sessions"]
)

print("\n=== ENTRY-TIME FEATURE ABLATION ===")

print(
    ablation_summary
    .round(4)
    .to_string()
)

print(
    "The fixed final-model threshold may produce different "
    "contact rates across feature sets. "
    "Interpret probability metrics and policy metrics together."
)



/Users/gonzalo/daiist-assignment-1/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/gonzalo/daiist-assignment-1/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/gonzalo/daiist-assignment-1/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/gonzalo/daiist-assignment-1/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1, 5] during transfor


=== ENTRY-TIME FEATURE ABLATION ===
                              total_cost  total_sessions  mean_monthly_cost  mean_contact_rate  mean_recall  mean_roc_auc  mean_log_loss  pooled_cost_per_session
feature_set                                                                                                                                                      
A: Entry baseline                 5301.0            4860             0.9522             0.0838       0.1167        0.6108         0.5036                   1.0907
B: + Cyclical calendar            5251.0            4860             0.9508             0.1009       0.1294        0.6133         0.5019                   1.0805
C: + Visitor interactions         5187.0            4860             0.9462             0.1064       0.1367        0.6118         0.5007                   1.0673
D: + Traffic/visitor segment      5206.0            4860             0.9483             0.1141       0.1426        0.6119         0.5001                 

/Users/gonzalo/daiist-assignment-1/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


## 18. Business-Cost Sensitivity Analysis

In [35]:

# Business-cost sensitivity analysis.
# The selected operating threshold remains FIXED.
# This does not select a different threshold or retrain a model.

cost_sensitivity_rows = []

for missed_buyer_penalty in [
    0.5, 1.0, 2.0, 5.0, 10.0
]:

    for month in VALIDATION_MONTHS:

        fold = fold_predictions[month]

        y = fold["actual"]

        contacted = (
            fold["probabilities"] >= selected_threshold
        )

        missed_buyers = int(np.sum(
            (~contacted) & (y == 1)
        ))

        n_contacted = int(np.sum(contacted))

        illustrative_cost = (
            ASSISTANCE_COST * n_contacted
            + missed_buyer_penalty * missed_buyers
        )

        cost_sensitivity_rows.append({
            "missed_buyer_penalty": missed_buyer_penalty,
            "month": month,
            "contact_rate": n_contacted / len(y),
            "illustrative_cost_per_session": (
                illustrative_cost / len(y)
            ),
        })


cost_sensitivity_df = pd.DataFrame(
    cost_sensitivity_rows
)

print(
    "\n=== HISTORICAL COST-ASSUMPTION SENSITIVITY ==="
)

print(
    cost_sensitivity_df
    .groupby("missed_buyer_penalty")[
        "illustrative_cost_per_session"
    ]
    .mean()
    .round(4)
)




=== HISTORICAL COST-ASSUMPTION SENSITIVITY ===
missed_buyer_penalty
0.5     0.1976
1.0     0.2810
2.0     0.4478
5.0     0.9483
10.0    1.7824
Name: illustrative_cost_per_session, dtype: float64


## 19. Final Holdout Evaluation

In [36]:
# Final test-set probabilities
# The threshold has already been chosen using validation data.

sklearn_test_prob = sklearn_model.predict_proba(X_test)[:, 1]

with torch.no_grad():
    manual_test_logits = (
        X_test_tensor @ manual_weights + manual_bias
    )

    manual_test_prob = torch.sigmoid(
        manual_test_logits
    ).squeeze().numpy()

torch_model.eval()

with torch.no_grad():
    torch_test_logits = torch_model(X_test_tensor)

    torch_test_prob = torch.sigmoid(
        torch_test_logits
    ).squeeze().numpy()

In [37]:
# Final naive baseline evaluation,
# after all historical model and policy selection.

baseline_test_metrics = evaluate_model(
    y_test,
    baseline_test_prob,
    threshold=0.5,
)

# Test-set evaluation at the standard 0.50 threshold

sklearn_metrics = evaluate_model(
    y_test,
    sklearn_test_prob,
    threshold=0.5,
)

manual_metrics = evaluate_model(
    y_test,
    manual_test_prob,
    threshold=0.5,
)

torch_metrics = evaluate_model(
    y_test,
    torch_test_prob,
    threshold=0.5,
)


results = pd.DataFrame(
    [
        {
            "model": "Naive baseline",
            **baseline_test_metrics,
        },
        {
            "model": "Scikit-learn",
            **sklearn_metrics,
        },
        {
            "model": "Manual PyTorch",
            **manual_metrics,
        },
        {
            "model": "PyTorch nn.Module",
            **torch_metrics,
        },
    ]
)


display_columns = [
    "model",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "log_loss",
    "fp",
    "fn",
    "business_cost",
]


print("\nTest results at threshold 0.50:")

print(
    results[display_columns]
    .round(4)
    .to_string(index=False)
)


Test results at threshold 0.50:
            model  accuracy  precision  recall  f1  roc_auc  log_loss  fp  fn  business_cost
   Naive baseline    0.8749        0.0     0.0 0.0   0.5000    0.3769   0 216         1080.0
     Scikit-learn    0.8749        0.0     0.0 0.0   0.6391    0.3663   0 216         1080.0
   Manual PyTorch    0.8749        0.0     0.0 0.0   0.6391    0.3663   0 216         1080.0
PyTorch nn.Module    0.8749        0.0     0.0 0.0   0.6391    0.3663   0 216         1080.0


In [38]:
sklearn_business_metrics = evaluate_model(
    y_test,
    sklearn_test_prob,
    threshold=selected_threshold,
)

manual_business_metrics = evaluate_model(
    y_test,
    manual_test_prob,
    threshold=selected_threshold,
)

torch_business_metrics = evaluate_model(
    y_test,
    torch_test_prob,
    threshold=selected_threshold,
)

In [39]:
business_results = pd.DataFrame(
    [
        {
            "model": "Scikit-learn",
            **sklearn_business_metrics,
        },
        {
            "model": "Manual PyTorch",
            **manual_business_metrics,
        },
        {
            "model": "PyTorch nn.Module",
            **torch_business_metrics,
        },
    ]
)

print("\nFinal policy:", selected_policy)
print(f"Selected threshold: {selected_threshold:.4f}")

print(
    business_results[
        [
            "model",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "fp",
            "fn",
            "business_cost",
        ]
    ].round(4).to_string(index=False)
)


Final policy: A: Conservative fixed cutoff
Selected threshold: 0.2500
            model  accuracy  precision  recall     f1  fp  fn  business_cost
     Scikit-learn    0.8321     0.2279  0.1435 0.1761 105 185         1061.0
   Manual PyTorch    0.8321     0.2279  0.1435 0.1761 105 185         1061.0
PyTorch nn.Module    0.8321     0.2279  0.1435 0.1761 105 185         1061.0


In [40]:
policy_baselines = pd.DataFrame([
    {
        "model": "Always negative",
        **evaluate_model(
            y_test,
            np.zeros(len(y_test)),
            threshold=selected_threshold,
        ),
    },
    {
        "model": "Always positive",
        **evaluate_model(
            y_test,
            np.ones(len(y_test)),
            threshold=selected_threshold,
        ),
    },
    {
        "model": "Historical-rate constant",
        **evaluate_model(
            y_test,
            baseline_test_prob,
            threshold=selected_threshold,
        ),
    },
])

business_results = pd.concat(
    [policy_baselines, business_results],
    ignore_index=True,
)

print(
    business_results[
        ["model", "roc_auc", "precision", "recall",
         "fp", "fn", "business_cost"]
    ].round(4).to_string(index=False)
)

                   model  roc_auc  precision  recall   fp  fn  business_cost
         Always negative   0.5000     0.0000  0.0000    0 216         1080.0
         Always positive   0.5000     0.1251  1.0000 1511   0         1727.0
Historical-rate constant   0.5000     0.0000  0.0000    0 216         1080.0
            Scikit-learn   0.6391     0.2279  0.1435  105 185         1061.0
          Manual PyTorch   0.6391     0.2279  0.1435  105 185         1061.0
       PyTorch nn.Module   0.6391     0.2279  0.1435  105 185         1061.0


In [41]:

# Audit the frozen entry-time policy on December.
# No threshold or selection rule is learned from December.

policy_audit = []

for name, probabilities in [
    ("Scikit-learn", sklearn_test_prob),
    ("Manual PyTorch", manual_test_prob),
    ("PyTorch nn.Module", torch_test_prob),
]:

    metrics = evaluate_model(
        y_test,
        probabilities,
        threshold=selected_threshold,
    )

    policy_audit.append({
        "model": name,
        "threshold": selected_threshold,
        "policy": selected_policy,

        "sessions_contacted": metrics["contacted"],
        "contact_rate": metrics["contact_rate"],

        "purchasers_contacted": metrics["tp"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],

        "capacity_exceeded": (
            metrics["contact_rate"]
            > ASSISTANCE_CAPACITY
        ),
    })


policy_audit_df = pd.DataFrame(policy_audit)

print("\n=== DECEMBER ENTRY-TIME POLICY AUDIT ===")

print(
    policy_audit_df.round(4).to_string(index=False)
)



=== DECEMBER ENTRY-TIME POLICY AUDIT ===
            model  threshold                       policy  sessions_contacted  contact_rate  purchasers_contacted  precision  recall  capacity_exceeded
     Scikit-learn       0.25 A: Conservative fixed cutoff                 136        0.0787                    31     0.2279  0.1435              False
   Manual PyTorch       0.25 A: Conservative fixed cutoff                 136        0.0787                    31     0.2279  0.1435              False
PyTorch nn.Module       0.25 A: Conservative fixed cutoff                 136        0.0787                    31     0.2279  0.1435              False


In [ ]:

# ============================================================
# FINAL DECEMBER COMPARISON OF THE THREE FROZEN POLICIES
# All thresholds were fixed using pre-December data.
# Use scikit-learn probabilities to isolate policy effects.
# ============================================================

december_policy_rows = []

for policy_name, threshold in final_policy_thresholds.items():

    metrics = evaluate_model(
        y_test,
        sklearn_test_prob,
        threshold=threshold,
    )

    december_policy_rows.append({
        "policy": policy_name,
        "threshold": threshold,

        "sessions_contacted": metrics["contacted"],
        "contact_rate": metrics["contact_rate"],

        "purchasers_contacted": metrics["tp"],
        "purchasers_missed": metrics["fn"],

        "precision": metrics["precision"],
        "recall": metrics["recall"],

        "illustrative_cost": metrics["business_cost"],
        "cost_per_session": (
            metrics["business_cost"] / len(y_test)
        ),

        "capacity_exceeded": (
            metrics["contact_rate"] > ASSISTANCE_CAPACITY
        ),
    })


december_policy_df = pd.DataFrame(
    december_policy_rows
)

print("\n=== DECEMBER: ALL THREE FROZEN POLICIES ===")

print(
    december_policy_df
    .round(4)
    .to_string(index=False)
)

print("\nPolicy chosen BEFORE December:", selected_policy)


# ------------------------------------------------------------
# OPTIONAL RETROSPECTIVE RANKING DIAGNOSTIC
# This is NOT a deployable entry-time policy.
# ------------------------------------------------------------

n_top = int(
    np.floor(ASSISTANCE_CAPACITY * len(y_test))
)

ranked_indices = np.argsort(
    -sklearn_test_prob,
    kind="stable",
)

top_indices = ranked_indices[:n_top]

retrospective_contacts = np.zeros(
    len(y_test),
    dtype=bool,
)

retrospective_contacts[top_indices] = True

retrospective_tp = int(np.sum(
    retrospective_contacts & (y_test == 1)
))

retrospective_precision = (
    retrospective_tp / n_top
    if n_top > 0 else 0.0
)

retrospective_recall = (
    retrospective_tp / int(np.sum(y_test))
)

print("RETROSPECTIVE TOP-20% RANKING DIAGNOSTIC")

print("Visitors selected:", n_top)

print("Eventual purchasers among them:", retrospective_tp)

print(
    f"Retrospective precision: "
    f"{retrospective_precision:.4f}"
)

print(
    f"Retrospective recall: "
    f"{retrospective_recall:.4f}"
)

print(
    "This diagnostic ranks all December visitors after "
    "their scores are available. It must not be treated "
    "as a live policy or used to choose the final model."
)



=== DECEMBER: ALL THREE FROZEN POLICIES ===
                         policy  threshold  sessions_contacted  contact_rate  purchasers_contacted  purchasers_missed  precision  recall  illustrative_cost  cost_per_session  capacity_exceeded
   A: Conservative fixed cutoff     0.2500                 136        0.0787                    31                185     0.2279  0.1435             1061.0            0.6144              False
B: Historical capacity quantile     0.2207                 257        0.1488                    50                166     0.1946  0.2315             1087.0            0.6294              False
       C: Cost floor + capacity     0.2207                 257        0.1488                    50                166     0.1946  0.2315             1087.0            0.6294              False

Policy chosen BEFORE December: A: Conservative fixed cutoff

=== RETROSPECTIVE TOP-20% RANKING DIAGNOSTIC ===
Visitors selected: 345
Eventual purchasers among them: 70
Retrospective 

## 20. Cross-Implementation Checks

In [ ]:
print("Maximum probability differences:")

print("sklearn vs manual:", np.max(np.abs(sklearn_test_prob - manual_test_prob)))
print("sklearn vs nn.Module:", np.max(np.abs(sklearn_test_prob - torch_test_prob)))
print("manual vs nn.Module:", np.max(np.abs(manual_test_prob - torch_test_prob)))

Maximum probability differences:
sklearn vs manual: 0.00013171050527677197
sklearn vs nn.Module: 0.00013171050527677197
manual vs nn.Module: 0.0


In [44]:
assert np.allclose(
    manual_test_prob,
    torch_test_prob,
    atol=1e-6,
), "Manual PyTorch and nn.Module PyTorch predictions differ unexpectedly."

## 21. Save Models and Artifacts

In [ ]:
joblib.dump(preprocessor, ARTIFACT_DIR / "preprocessor.joblib",)
joblib.dump(sklearn_model, ARTIFACT_DIR / "sklearn_model.joblib",)

['artifacts/sklearn_model.joblib']

In [46]:
torch.save(
    {
        "weights": manual_weights.detach().cpu(),
        "bias": manual_bias.detach().cpu(),
        "n_features": n_features,
    },
    ARTIFACT_DIR / "manual_pytorch_model.pt",
)

In [47]:
torch.save(
    {
        "state_dict": torch_model.state_dict(),
        "n_features": n_features,
    },
    ARTIFACT_DIR / "pytorch_module_model.pt",
)

In [ ]:
np.save(ARTIFACT_DIR / "X_test.npy", X_test,)
np.save(ARTIFACT_DIR / "y_test.npy", y_test,)

In [ ]:
# Preserve descriptive test data for the dashboard.
# Only metadata-listed entry-time features are used for inference.

dashboard_test_data = test_df.copy()

dashboard_test_data.to_csv(ARTIFACT_DIR / "dashboard_test_data.csv", index=False,)

In [ ]:
prediction_data = pd.DataFrame(
    {
        "actual": y_test,
        "sklearn_probability": sklearn_test_prob,
        "manual_probability": manual_test_prob,
        "torch_probability": torch_test_prob,
    }
)

prediction_data.to_csv(ARTIFACT_DIR / "test_predictions.csv", index=False,)

In [ ]:
results.to_csv(ARTIFACT_DIR / "metrics_threshold_050.csv", index=False,)
business_results.to_csv(ARTIFACT_DIR / "metrics_business_threshold.csv", index=False,)
rolling_results.to_csv(ARTIFACT_DIR / "threshold_analysis.csv", index=False,)
threshold_df.to_csv(ARTIFACT_DIR / "november_threshold_analysis.csv", index=False,)

In [ ]:
ablation_df.to_csv(ARTIFACT_DIR / "feature_ablation_folds.csv", index=False,)
ablation_summary.to_csv(ARTIFACT_DIR / "feature_ablation_summary.csv",)
regularization_df.to_csv(ARTIFACT_DIR / "regularization_results.csv", index=False,)
cost_sensitivity_df.to_csv(ARTIFACT_DIR / "cost_sensitivity.csv", index=False,)
policy_audit_df.to_csv(ARTIFACT_DIR / "capacity_policy_audit.csv", index=False,)
policy_replay_df.to_csv(ARTIFACT_DIR / "policy_replay_monthly.csv", index=False,)
policy_summary_df.to_csv(ARTIFACT_DIR / "policy_replay_summary.csv", index=False,)
december_policy_df.to_csv(ARTIFACT_DIR / "december_policy_comparison.csv", index=False,)

In [ ]:
metadata = {
    "target": TARGET,

    "prediction_moment": (
        "Beginning of browsing session, "
        "before session activity accumulates"
    ),

    "assistance_capacity_fraction": (
        ASSISTANCE_CAPACITY
    ),

    "best_threshold": selected_threshold,

    "selected_policy": selected_policy,

    "policy_thresholds": final_policy_thresholds,

    "policy_comparison_months": [
        "September",
        "October",
        "November",
    ],

    "policy_capacity_warning": (
        "A historically fitted probability threshold "
        "does not guarantee future capacity compliance."
    ),

    "assistance_cost": ASSISTANCE_COST,

    "missed_buyer_service_cost": (
        MISSED_BUYER_SERVICE_COST
    ),

    "true_positive_cost": TP_COST,
    "false_positive_cost": FP_COST,
    "false_negative_cost": FN_COST,

    "business_cost_formula": (
        "assistance_cost * (TP + FP) "
        "+ missed_buyer_service_cost * FN"
    ),

    "cost_units": (
        "Hypothetical service-level units; "
        "not causal lost-sales estimates"
    ),

    "seed": SEED,

    "manual_learning_rate": MANUAL_LR,
    "manual_epochs": MANUAL_EPOCHS,

    "torch_learning_rate": TORCH_LR,
    "torch_epochs": TORCH_EPOCHS,

    "model_feature_names": feature_names.tolist(),
    "n_model_features": int(n_features),

    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,

    "regularization_C": SKLEARN_C,

    "threshold_selection": (
        "Three policies compared through historical "
        "walk-forward replay on September-November, "
        "using only previous months to fit each cutoff. "
        "Prefer policies satisfying the 20% contact "
        "constraint in every replay month, then maximize "
        "mean monthly purchaser recall. Freeze the "
        "selected policy and its cutoff before December."
    ),

    "train_period": "February to October",
    "validation_period": "November",
    "test_period": "December",
}

with (
    ARTIFACT_DIR / "metadata.json"
).open("w", encoding="utf-8") as f:

    json.dump(metadata, f, indent=2)


In [54]:
print("\nSaved artifacts:")

for path in sorted(ARTIFACT_DIR.iterdir()):
    print("-", path.name)


Saved artifacts:
- X_test.npy
- capacity_policy_audit.csv
- cost_sensitivity.csv
- dashboard_test_data.csv
- december_policy_comparison.csv
- feature_ablation_folds.csv
- feature_ablation_summary.csv
- manual_pytorch_model.pt
- manual_training_losses.png
- metadata.json
- metrics_business_threshold.csv
- metrics_threshold_050.csv
- november_threshold_analysis.csv
- policy_replay_monthly.csv
- policy_replay_summary.csv
- preprocessor.joblib
- pytorch_module_model.pt
- regularization_results.csv
- sklearn_model.joblib
- test_predictions.csv
- threshold_analysis.csv
- y_test.npy
